Importing the libaries


In [1]:
# Importing the necessary libraries

# Enable automatic reloading of modules when they are updated
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import os
import textwrap

PROJECT_ROOT = Path.cwd().resolve().parent
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))



import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from IPython.display import Image, display
from torchmetrics.text.bleu import BLEUScore

from src.train import (
    build_sequence_dataloaders,
    build_tokenizer,
    load_storyreasoning,
    train_sequence_predictor,
    train_experiment2,
)

from src.utils import (
    ensure_dirs,
    generate,
    load_config,
    set_seed,
    validation,
)

CONFIG_PATH = PROJECT_ROOT / "config.yaml"
config = load_config(str(CONFIG_PATH))


set_seed(config.get("seed", 42))
ensure_dirs(config["paths"]["checkpoint_dir"], config["paths"]["results_dir"])

output_dir = Path(config["paths"]["results_dir"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Project root: {PROJECT_ROOT}")
print(f"Using device: {device}")


Project root: C:\Users\Kishan Prasad Jasiwa\Desktop\Dnnls_Final_Project\dnnls_final_project
Using device: cuda


Loading and Saving Data

In [39]:
# Loading the dataset
tokenizer = build_tokenizer()
train_dataset, test_dataset = load_storyreasoning(config)
train_dataloader, val_dataloader, test_dataloader = build_sequence_dataloaders(
    config,
    tokenizer,
    train_dataset,
    test_dataset,
)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Train batches: {len(train_dataloader)}")
print(f"Validation batches: {len(val_dataloader)}")
print(f"Test batches: {len(test_dataloader)}")


Train samples: 3552
Test samples: 626
Train batches: 356
Validation batches: 178
Test batches: 157


In [40]:
"""
A sanity check cell to verify the data pipeline.
It grabs a single batch from the training dataset and prints the shapes of the returned tensors (images, descriptions, etc.) to ensure everything is loaded correctly.
"""

frames, descriptions, image_target, text_target, roi1, roi2, roi_valid, roi_frame, ent_id = next(iter(train_dataloader))

print("frames:", frames.shape)
print("descriptions:", descriptions.shape)
print("image_target:", image_target.shape)
print("text_target:", text_target.shape)
print("roi_valid:", roi_valid.shape)


frames: torch.Size([8, 4, 3, 60, 125])
descriptions: torch.Size([8, 4, 120])
image_target: torch.Size([8, 3, 60, 125])
text_target: torch.Size([8, 1, 120])
roi_valid: torch.Size([8])


Experiment 2 - Sequence Predictor: Bidirectional GRU

In [ ]:
experiment_name = "Experiment_2"
experiment2_dir = Path("results") / experiment_name
output_dir = experiment2_dir
ensure_dirs(experiment2_dir)
print(f"Experiment 2 outputs: {experiment2_dir}") 

'\nexperiment_name = "Experiment_2"\nexperiment2_dir = Path("results") / experiment_name\noutput_dir = experiment2_dir\nensure_dirs(experiment2_dir)\nprint(f"Experiment 2 outputs: {experiment2_dir}") \n'

Training

In [ ]:
sequence_predictor, tokenizer, val_dataloader, losses, training_log = train_experiment2(
    CONFIG_PATH,
    show_validation=True,
)

'\nsequence_predictor, tokenizer, val_dataloader, losses, training_log = train_experiment2(\n    CONFIG_PATH,\n    show_validation=True,\n)\n'

Saving the Training logs

In [ ]:
experiment2_dir = Path("results") / "Experiment_2"
output_dir = experiment2_dir
ensure_dirs(experiment2_dir)
log_path = experiment2_dir / "training_log.txt"

with open(log_path, "w", encoding="utf-8") as f:
    for line in training_log:
        print(line)
        f.write(line + "\n")

print(f"Training log saved: {log_path}")

'\nexperiment2_dir = Path("results") / "Experiment_2"\noutput_dir = experiment2_dir\nensure_dirs(experiment2_dir)\nlog_path = experiment2_dir / "training_log.txt"\n\nwith open(log_path, "w", encoding="utf-8") as f:\n    for line in training_log:\n        print(line)\n        f.write(line + "\n")\n\nprint(f"Training log saved: {log_path}")\n'

Validation Run

In [ ]:
validation(
    sequence_predictor,
    val_dataloader,
    tokenizer,
    device,
    show=True,
)
sequence_predictor.eval()

'\nvalidation(\n    sequence_predictor,\n    val_dataloader,\n    tokenizer,\n    device,\n    show=True,\n)\nsequence_predictor.eval()\n'

Image Output Visualization

In [ ]:
pred_path = experiment2_dir / "predictionexample.png"
gt_path = experiment2_dir / "groundtruth.png"
comparison_path = experiment2_dir / "visualcomparison.png"

sequence_predictor.eval()
frames, descriptions, image_target, text_target, *_ = next(iter(val_dataloader))
frames = frames.to(device)
descriptions = descriptions.to(device)
image_target = image_target.to(device)
text_target = text_target.to(device)

with torch.no_grad():
    pred_img, _, _, h0, c0, _, _ = sequence_predictor(frames, descriptions, text_target)
    generated_tokens = generate(
        sequence_predictor.text_decoder,
        h0[:, 0, :].unsqueeze(1),
        c0[:, 0, :].unsqueeze(1),
        max_len=150,
        sos_token_id=tokenizer.cls_token_id,
        eos_token_id=tokenizer.sep_token_id,
        device=device,
    )

if text_target.dim() == 3:
    text_target_decode = text_target.squeeze(1)
else:
    text_target_decode = text_target

true_sentence = tokenizer.decode(text_target_decode[0].cpu(), skip_special_tokens=True)
pred_sentence = tokenizer.decode(generated_tokens, skip_special_tokens=True)

plt.imsave(pred_path, pred_img[0].detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy())
plt.imsave(gt_path, image_target[0].detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy())

# Side by Side comparision figure
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(image_target[0].detach().cpu().clamp(0, 1).permute(1, 2, 0))
plt.title("Ground Truth (Target)")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(pred_img[0].detach().cpu().clamp(0, 1).permute(1, 2, 0))
plt.title("Experiment 2 Prediction")
plt.axis("off")

plt.tight_layout()
plt.savefig(comparison_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()

print("Target Text:", true_sentence)
print("Predicted Text:", pred_sentence)
print(f"Saved: {pred_path}, {gt_path} and {comparison_path}")


'\npred_path = experiment2_dir / "predictionexample.png"\ngt_path = experiment2_dir / "groundtruth.png"\ncomparison_path = experiment2_dir / "visualcomparison.png"\n\nsequence_predictor.eval()\nframes, descriptions, image_target, text_target, *_ = next(iter(val_dataloader))\nframes = frames.to(device)\ndescriptions = descriptions.to(device)\nimage_target = image_target.to(device)\ntext_target = text_target.to(device)\n\nwith torch.no_grad():\n    pred_img, _, _, h0, c0, _, _ = sequence_predictor(frames, descriptions, text_target)\n    generated_tokens = generate(\n        sequence_predictor.text_decoder,\n        h0[:, 0, :].unsqueeze(1),\n        c0[:, 0, :].unsqueeze(1),\n        max_len=150,\n        sos_token_id=tokenizer.cls_token_id,\n        eos_token_id=tokenizer.sep_token_id,\n        device=device,\n    )\n\nif text_target.dim() == 3:\n    text_target_decode = text_target.squeeze(1)\nelse:\n    text_target_decode = text_target\n\ntrue_sentence = tokenizer.decode(text_target_d

Saliencey Map

In [ ]:
saliency_path = experiment2_dir / "saliencymap.png"
sequence_predictor.train()
frames, descriptions, image_target, text_target, *_ = next(iter(val_dataloader))
frames = frames.to(device).clone().detach().requires_grad_(True)
descriptions = descriptions.to(device)
image_target = image_target.to(device)
text_target = text_target.to(device)

sequence_predictor.zero_grad(set_to_none=True)
pred_img, _, pred_text_logits, _, _, _, _ = sequence_predictor(frames, descriptions, text_target)

prediction_flat = pred_text_logits.reshape(-1, tokenizer.vocab_size)
target_labels = text_target.squeeze(1)[:, 1:]
target_flat = target_labels.reshape(-1)

saliency_loss = F.l1_loss(pred_img, image_target) + F.cross_entropy(
    prediction_flat,
    target_flat,
    ignore_index=tokenizer.convert_tokens_to_ids(tokenizer.pad_token),
)
saliency_loss.backward()

frame_saliency = frames.grad.detach().abs().mean(dim=(2, 3, 4))[0]
frame_saliency = (frame_saliency - frame_saliency.min()) / (frame_saliency.max() - frame_saliency.min() + 1e-8)

sequence_predictor.eval()

plt.figure(figsize=(7, 2.2))
heatmap = plt.imshow(frame_saliency.unsqueeze(0).cpu().numpy(), cmap="YlOrRd", aspect="auto", vmin=0.0, vmax=1.0)
plt.title("Experiment 2: Frame Saliency Map")
plt.xlabel("Input Frame")
plt.yticks([])
plt.xticks(range(frame_saliency.numel()), [f"Frame {i + 1}" for i in range(frame_saliency.numel())])

for col, value in enumerate(frame_saliency.cpu().tolist()):
    text_color = "white" if value > 0.55 else "black"
    plt.text(col, 0, f"{value:.2f}", ha="center", va="center", fontsize=10, color=text_color)

plt.colorbar(heatmap, fraction=0.035, pad=0.02, label="Saliency Strength")
plt.tight_layout()
plt.savefig(saliency_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print(f"Saliency Map Saved: {saliency_path}")

'\nsaliency_path = experiment2_dir / "saliencymap.png"\n\n# CuDNN RNN backward needs training mode for saliency gradients.\nsequence_predictor.train()\nframes, descriptions, image_target, text_target, *_ = next(iter(val_dataloader))\nframes = frames.to(device).clone().detach().requires_grad_(True)\ndescriptions = descriptions.to(device)\nimage_target = image_target.to(device)\ntext_target = text_target.to(device)\n\nsequence_predictor.zero_grad(set_to_none=True)\npred_img, _, pred_text_logits, _, _, _, _ = sequence_predictor(frames, descriptions, text_target)\n\nprediction_flat = pred_text_logits.reshape(-1, tokenizer.vocab_size)\ntarget_labels = text_target.squeeze(1)[:, 1:]\ntarget_flat = target_labels.reshape(-1)\n\nsaliency_loss = F.l1_loss(pred_img, image_target) + F.cross_entropy(\n    prediction_flat,\n    target_flat,\n    ignore_index=tokenizer.convert_tokens_to_ids(tokenizer.pad_token),\n)\nsaliency_loss.backward()\n\nframe_saliency = frames.grad.detach().abs().mean(dim=(2, 3

Loss Curve

In [ ]:
plot_path = experiment2_dir / "losscurve.png"

plt.figure(figsize=(8, 5))
plt.plot(losses, label="Experiment 2 Training Loss", color="blue", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Experiment 2: Loss Curve")
plt.legend()
plt.grid(True)

plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()

print(f"Loss Curve Saved: {plot_path}")

'\nplot_path = experiment2_dir / "losscurve.png"\n\nplt.figure(figsize=(8, 5))\nplt.plot(losses, label="Experiment 2 Training Loss", color="blue", linewidth=2)\nplt.xlabel("Epoch")\nplt.ylabel("Loss")\nplt.title("Experiment 2: Loss Curve")\nplt.legend()\nplt.grid(True)\n\nplt.savefig(plot_path, dpi=150, bbox_inches="tight")\nplt.show()\nplt.close()\n\nprint(f"Loss Curve Saved: {plot_path}")\n'

Metrics Calculation & Table Saving

In [ ]:
sequence_predictor.eval()
pred_sentences = []
true_sentences = []
max_batches = config.get("evaluation", {}).get("bleu_max_batches", 5)

with torch.no_grad():
    for batch_index, (frames, descriptions, image_target, text_target, *_ ) in enumerate(val_dataloader):
        if max_batches is not None and batch_index >= max_batches:
            break

        frames = frames.to(device)
        descriptions = descriptions.to(device)
        text_target = text_target.to(device)

        _, _, _, h0, c0, _, _ = sequence_predictor(frames, descriptions, text_target)

        # Generate predictions
        for i in range(frames.size(0)):
            generated_tokens = generate(
                sequence_predictor.text_decoder,
                h0[:, i, :].unsqueeze(1),
                c0[:, i, :].unsqueeze(1),
                max_len=150,
                sos_token_id=tokenizer.cls_token_id,
                eos_token_id=tokenizer.sep_token_id,
                device=device,
            )
            pred_sentences.append(tokenizer.decode(generated_tokens, skip_special_tokens=True))

        # Ground truth
        if text_target.dim() == 3:
            text_target_decode = text_target.squeeze(1)
        else:
            text_target_decode = text_target

        for seq in text_target_decode:
            true_sentences.append(tokenizer.decode(seq.cpu().numpy(), skip_special_tokens=True))

# Calculate BLEU-4 Score
bleu_metric = BLEUScore(n_gram=4)
reference = [[s] for s in true_sentences]
experiment2_bleu_val = bleu_metric(pred_sentences, reference).item()

# Results
print("\n" + "=" * 50)
print("Experiment 2 Results")
print("=" * 50)
print(f"Final Training Loss: {losses[-1]:.4f}")
print(f"BLEU-4 Accuracy: {experiment2_bleu_val:.4f}")
print("=" * 50 + "\n")

# Save to metrics.txt
metrics_path = experiment2_dir / "metrics.txt"
with open(metrics_path, "w", encoding="utf-8") as f:
    f.write("Experiment 2\n")
    f.write("=" * 40 + "\n")
    f.write(f"{'Metric':<25} | {'Value':<10}\n")
    f.write("-" * 40 + "\n")
    f.write(f"{'Final Training Loss':<25} | {losses[-1]:.4f}\n")
    f.write(f"{'BLEU-4 Accuracy':<25} | {experiment2_bleu_val:.4f}\n")
    f.write(f"{'Epochs Completed':<25} | {len(losses)}\n")
    f.write(f"{'Predictions Evaluated':<25} | {len(pred_sentences)}\n")

print(f"Experiment 2 metrics table saved: {metrics_path}")
print("Number of predictions:", len(pred_sentences))
print("Number of true sentences:", len(true_sentences))

'\nsequence_predictor.eval()\npred_sentences = []\ntrue_sentences = []\nmax_batches = config.get("evaluation", {}).get("bleu_max_batches", 5)\n\nwith torch.no_grad():\n    for batch_index, (frames, descriptions, image_target, text_target, *_ ) in enumerate(val_dataloader):\n        if max_batches is not None and batch_index >= max_batches:\n            break\n\n        frames = frames.to(device)\n        descriptions = descriptions.to(device)\n        text_target = text_target.to(device)\n\n        _, _, _, h0, c0, _, _ = sequence_predictor(frames, descriptions, text_target)\n\n        # Generate predictions\n        for i in range(frames.size(0)):\n            generated_tokens = generate(\n                sequence_predictor.text_decoder,\n                h0[:, i, :].unsqueeze(1),\n                c0[:, i, :].unsqueeze(1),\n                max_len=150,\n                sos_token_id=tokenizer.cls_token_id,\n                eos_token_id=tokenizer.sep_token_id,\n                device=dev

Comparison

In [ ]:
baseline_dir = Path("results") / "baseline"
experiment2_dir = Path("results") / "Experiment_2"
comparison_path = experiment2_dir / "comparison_table.txt"

baseline_metrics = {}
with open(baseline_dir / "metrics.txt", "r", encoding="utf-8") as f:
    for line in f:
        if "|" in line and "Metric" not in line:
            name, value = line.split("|", 1)
            name = name.strip()
            value = value.strip()
            try:
                baseline_metrics[name] = float(value)
            except ValueError:
                baseline_metrics[name] = value

experiment2_metrics = {}
with open(experiment2_dir / "metrics.txt", "r", encoding="utf-8") as f:
    for line in f:
        if "|" in line and "Metric" not in line:
            name, value = line.split("|", 1)
            name = name.strip()
            value = value.strip()
            try:
                experiment2_metrics[name] = float(value)
            except ValueError:
                experiment2_metrics[name] = value

comparison_rows = [
    "Experiment 2 vs Baseline",
    "=" * 56,
    f"{'Metric':<25} | {'Baseline':<10} | {'Exp 2':<10} | {'Change':<10}",
    "-" * 56,
]

for metric in ["Final Training Loss", "BLEU-4 Accuracy", "Epochs Completed", "Predictions Evaluated"]:
    baseline_value = baseline_metrics.get(metric, 0.0)
    experiment2_value = experiment2_metrics.get(metric, 0.0)
    if isinstance(baseline_value, float) and isinstance(experiment2_value, float):
        change = experiment2_value - baseline_value
        comparison_rows.append(f"{metric:<25} | {baseline_value:<10.4f} | {experiment2_value:<10.4f} | {change:<10.4f}")
    else:
        comparison_rows.append(f"{metric:<25} | {baseline_value!s:<10} | {experiment2_value!s:<10} | {'-':<10}")

comparison_text = "\n".join(comparison_rows)
print(comparison_text)

with open(comparison_path, "w", encoding="utf-8") as f:
    f.write(comparison_text + "\n")

print(f"Comparison table saved: {comparison_path}")

'\nbaseline_dir = Path("results") / "baseline"\nexperiment2_dir = Path("results") / "Experiment_2"\ncomparison_path = experiment2_dir / "comparison_table.txt"\n\nbaseline_metrics = {}\nwith open(baseline_dir / "metrics.txt", "r", encoding="utf-8") as f:\n    for line in f:\n        if "|" in line and "Metric" not in line:\n            name, value = line.split("|", 1)\n            name = name.strip()\n            value = value.strip()\n            try:\n                baseline_metrics[name] = float(value)\n            except ValueError:\n                baseline_metrics[name] = value\n\nexperiment2_metrics = {}\nwith open(experiment2_dir / "metrics.txt", "r", encoding="utf-8") as f:\n    for line in f:\n        if "|" in line and "Metric" not in line:\n            name, value = line.split("|", 1)\n            name = name.strip()\n            value = value.strip()\n            try:\n                experiment2_metrics[name] = float(value)\n            except ValueError:\n               